# Türkiye Üniversiteleri Arası Bilimsel İş Birliği Ağı
## Sosyal Ağ Analizi — Final Raporu ve İleri Analiz Hattı (30 Ekim 2026)

**Kurum:** Manisa Celal Bayar Üniversitesi — Büyük Veri Analistliği
**Proje Kapsamı:** OpenAlex 2020–2025 dönemi Türkiye Bilgisayar Bilimi (Computer Science) yayınları üzerinden kurumsal iş birliği ağı.

Bu notebook, 30 Ekim 2026 final teslim ödev kartında tanımlanan tüm analiz adımlarını modüler betikler (`scripts/`) aracılığıyla baştan sona çalıştırır ve sonuçları görselleştirir:
1. **Temel Ağ Ölçüleri ve Merkeziyetler:** Kapsam B1 (186 aktif üniversite, tek bileşen) ve Kapsam B (194 üniversite, duyarlılık analizi), 5 merkeziyet ölçüsü, Spearman korelasyonu.
2. **Derece Dağılımı ve Güç Yasası:** `powerlaw` ile log-log CCDF grafiği, $\alpha, x_{min}$, lognormal ve üstel model karşılaştırmaları.
3. **Louvain ve Leiden Topluluk Tespiti:** 10 farklı sabit tohum ile kararlılık analizi (NMI ve ARI), en büyük toplulukların dökümü.
4. **Null Model Analizi:** Derece dizisini %100 koruyan 100 rastgele ağ ile $C, L, Q$ Z-skorları ve Small-Worldness ($\\sigma$).
5. **Zorunlu İleri Yöntem: Çekim (Gravity) Modeli Regresyonu:** 17.205 ikili çift, Haversine mesafesi ($d_{eff} = d + 1\text{ km}$), Poisson ve Negative Binomial GLM modelleri, aşırı yayılım kontrolü ve Ege Bölgesi çekim etkisi.
6. **Etkileşimli Coğrafi Harita:** Folium ile Türkiye geneli iş birliği haritası ve jitter ayrıştırması.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image

# Proje kök dizinini ekle
sys.path.append(os.path.abspath(".."))

from scripts.run_final_analysis import load_network_data, build_scope_graph
from scripts.analysis_metrics import compute_network_topology, compute_centralities
from scripts.analysis_powerlaw import fit_powerlaw_distribution
from scripts.analysis_community import analyze_communities, SEEDS_10
from scripts.analysis_null_model import run_null_model_experiments
from scripts.analysis_gravity import build_dyadic_dataset, fit_gravity_models
from scripts.visualize_map import generate_collaboration_map

print("Gerekli kütüphaneler ve analiz modülleri başarıyla yüklendi.")

## 1. Veri Yükleme ve Ağ Grafiği İnşası
İşlenmiş kurum metaverileri (`institutions.csv`) ve kenar listesi (`edges.csv`) yüklenir. Analizin ana odağı **Kapsam B1** (186 aktif üniversite) olup, **Kapsam B** (194 üniversite) duyarlılık kıyaslaması için kurulur.

In [ ]:
df_inst, df_edges = load_network_data("../data/processed/institutions.csv", "../data/processed/edges.csv")

# Kapsam B1: 186 Aktif Üniversite
G_b1, df_b1 = build_scope_graph(df_inst, df_edges, allowed_categories=["active_university"])

# Kapsam B: 194 Aktif ve Kapatılan Üniversite
G_b, df_b = build_scope_graph(df_inst, df_edges, allowed_categories=["active_university", "closed_2016_khk", "closed_2020_cbk"])

print(f"Scope B1 Düğüm Sayısı: {G_b1.number_of_nodes()}, Kenar Sayısı: {G_b1.number_of_edges()}")
print(f"Scope B  Düğüm Sayısı: {G_b.number_of_nodes()}, Kenar Sayısı: {G_b.number_of_edges()}")

## 2. Temel Ağ Ölçüleri ve Merkeziyetler
- Düğüm ve kenar sayıları, yoğunluk, kümelenme katsayısı, derece benzeşimi.
- Bağlantısız ağlarda tüm ağ ile dev bağlı bileşen (LCC) ayrımı.
- Degree, Weighted Degree, Betweenness, Closeness ve PageRank merkeziyetleri ile Spearman korelasyonu.

In [ ]:
topo_b1 = compute_network_topology(G_b1, scope_name="Scope B1")
topo_b = compute_network_topology(G_b, scope_name="Scope B")

df_topo = pd.DataFrame([
    {
        "Metrik": "Düğüm Sayısı (N)",
        "Scope B1 (Aktif Üniversiteler)": topo_b1["number_of_nodes"],
        "Scope B (Tüm Üniversiteler)": topo_b["number_of_nodes"]
    },
    {
        "Metrik": "Kenar Sayısı (M)",
        "Scope B1 (Aktif Üniversiteler)": topo_b1["number_of_edges"],
        "Scope B (Tüm Üniversiteler)": topo_b["number_of_edges"]
    },
    {
        "Metrik": "Ağ Yoğunluğu (Density)",
        "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["density"]:.4f}",
        "Scope B (Tüm Üniversiteler)": f"{topo_b["density"]:.4f}"
    },
    {
        "Metrik": "Bağlı Bileşen Sayısı",
        "Scope B1 (Aktif Üniversiteler)": topo_b1["number_connected_components"],
        "Scope B (Tüm Üniversiteler)": topo_b["number_connected_components"]
    },
    {
        "Metrik": "İzole Düğüm Sayısı",
        "Scope B1 (Aktif Üniversiteler)": topo_b1["number_isolated_nodes"],
        "Scope B (Tüm Üniversiteler)": topo_b["number_isolated_nodes"]
    },
    {
        "Metrik": "Ortalama Kümelenme (C)",
        "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["average_clustering_coefficient"]:.4f}",
        "Scope B (Tüm Üniversiteler)": f"{topo_b["average_clustering_coefficient"]:.4f}"
    },
    {
        "Metrik": "Derece Benzeşimi (r)",
        "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["degree_assortativity"]:.4f}",
        "Scope B (Tüm Üniversiteler)": f"{topo_b["degree_assortativity"]:.4f}"
    },
    {
        "Metrik": "Ortalama En Kısa Yol (Tüm Ağ)",
        "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["full_network_average_path_length"]:.4f}",
        "Scope B (Tüm Üniversiteler)": "Tanımsız (Bağlantısız Ağ)"
    },
    {
        "Metrik": "LCC Ortalama En Kısa Yol (L)",
        "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["largest_connected_component"]["average_shortest_path_length"]:.4f}",
        "Scope B (Tüm Üniversiteler)": f"{topo_b["largest_connected_component"]["average_shortest_path_length"]:.4f}"
    },
    {
        "Metrik": "Ağ Çapı (LCC Diameter)",
        "Scope B1 (Aktif Üniversiteler)": topo_b1["largest_connected_component"]["diameter"],
        "Scope B (Tüm Üniversiteler)": topo_b["largest_connected_component"]["diameter"]
    }
])
display(df_topo)

In [ ]:
centrality_df, top10_df, corr_matrix = compute_centralities(G_b1, inst_metadata_df=df_b1)

print("--- İlk 10 Kurum Karşılaştırması (Derece Merkeziliği) ---")
display(top10_df[top10_df["metric"] == "degree"].head(10))

print("\n--- Merkeziyetler ve Yayın Sayısı Arası Spearman Sıra Korelasyonu ---")
display(corr_matrix)

## 3. Derece Dağılımı ve Güç Yasası (Power Law) Analizi
- `powerlaw` kütüphanesi ile $\alpha, x_{min}$ parametreleri tahmin edilir.
- Üstel ve log-normal modellerle log-likelihood ratio testleri yürütülür.
- Küçük örneklem sınırlılığı ve dar derece aralığı nedeniyle saf güç yasası iddiası bilimsel olarak tartışılır.

In [ ]:
degrees = [d for _, d in G_b1.degree()]
pw_res = fit_powerlaw_distribution(degrees, output_plot_path="../reports/generated/degree_distribution_powerlaw.png")

print(f"Power Law Alpha: {pw_res["alpha"]:.4f}")
print(f"Minimum Eşik (xmin): {pw_res["xmin"]}")
print(f"Standart Hata (sigma): {pw_res["sigma"]:.4f}")
print(f"Üstel Modele Karşı R: {pw_res["comparison"]["vs_exponential"]["R"]:.3f} (p={pw_res["comparison"]["vs_exponential"]["p_value"]:.2e})")
print(f"Log-Normal Modele Karşı R: {pw_res["comparison"]["vs_lognormal"]["R"]:.3f} (p={pw_res["comparison"]["vs_lognormal"]["p_value"]:.4f})")
print(f"Saf Güç Yasası Destekleniyor mu?: {pw_res["comparison"]["supports_pure_power_law"]}")

# Üretilen grafiği göster
Image("../reports/generated/degree_distribution_powerlaw.png")

## 4. Topluluk Tespiti: Louvain ve Leiden Kararlılık Analizi
- 10 farklı sabit tohum (seed) ile tekrarlanabilir çalıştırma.
- NMI (Normalized Mutual Information) ve ARI (Adjusted Rand Index) metrikleriyle kararlılık ölçümü.
- En büyük 5 topluluğun hub kurumları ve Ege Bölgesi dağılımı.

In [ ]:
comm_res = analyze_communities(G_b1, inst_metadata_df=df_b1, seeds=SEEDS_10)

print(f"Louvain Ortalama Topluluk Sayısı: {comm_res["summary"]["louvain"]["num_communities_mean"]:.1f}, Q: {comm_res["summary"]["louvain"]["modularity_mean"]:.4f}")
print(f"Louvain Kararlılık: NMI={comm_res["summary"]["louvain"]["stability"]["nmi_mean"]:.4f}, ARI={comm_res["summary"]["louvain"]["stability"]["ari_mean"]:.4f}")
print(f"Leiden  Ortalama Topluluk Sayısı: {comm_res["summary"]["leiden"]["num_communities_mean"]:.1f}, Q: {comm_res["summary"]["leiden"]["modularity_mean"]:.4f}")
print(f"Leiden  Kararlılık: NMI={comm_res["summary"]["leiden"]["stability"]["nmi_mean"]:.4f}, ARI={comm_res["summary"]["leiden"]["stability"]["ari_mean"]:.4f}")
print(f"İki Algoritma Arası Mutabakat: NMI={comm_res["summary"]["cross_algorithm_agreement"]["nmi"]:.4f}, ARI={comm_res["summary"]["cross_algorithm_agreement"]["ari"]:.4f}")

print("\n--- En Büyük Topluluklar ve Hub Kurumları ---")
display(comm_res["top_clusters_df"])

## 5. Null Model Analizi (100 Derece Korumalı Rastgele Ağ)
- `connected_double_edge_swap` ile derece dizisi %100 korunan 100 rastgele ağ üretilir.
- Ampirik ağın kümelenme ($C$), yol uzunluğu ($L$) ve modülerlik ($Q$) değerleri null dağılımla kıyaslanır.
- Z-skoru, ampirik $p$-değeri ve Small-Worldness katsayısı ($\\sigma$) hesaplanır.

In [ ]:
# Hızlı interaktif görüntüleme için önceden üretilen 100 ağ sonuçları okunabilir ya da modülden çağrılabilir
null_summary_path = "../reports/generated/null_model_comparison.json"
with open(null_summary_path, "r", encoding="utf-8") as f:
    null_sum = json.load(f)

df_null = pd.read_csv("../reports/generated/null_model_comparison.csv")
display(df_null)

print(f"Küçük Dünya Katsayısı (sigma): {null_sum["small_worldness"]["sigma"]:.4f} (Small-World: {null_sum["small_worldness"]["is_small_world"]})")

## 6. Zorunlu İleri Yöntem: Çekim (Gravity) Modeli Regresyonu
- $\\binom{186}{2} = 17.205$ ikili üniversite çifti ($w=0$ sıfır iş birliği çiftleri eksiksiz dahil).
- Büyük daire (Haversine) mesafesi ve aynı şehir çakışmaları için $d_{eff} = d + 1.0\text{ km}$ düzeltmesi.
- Kurum büyüklüğü: 2020–2025 dönemi CS yayın hacmi (`period_cs_works_count`).
- Poisson ve Negative Binomial GLM modelleri, aşırı yayılım testi ve Ege Bölgesi ($BothEgean$) çekim faktörü.

In [ ]:
with open("../reports/generated/gravity_model_results.json", "r", encoding="utf-8") as f:
    grav_results = json.load(f)

df_grav_summary = pd.read_csv("../reports/generated/gravity_model_summary.csv")
display(df_grav_summary)

print("\n--- Aşırı Yayılım (Overdispersion) ve Model Tavsiyesi ---")
print(f"Pearson Dispersiyon Oranı: {grav_results["poisson_model"]["pearson_dispersion_ratio"]:.2f}")
print(f"Aşırı Yayılım Tespit Edildi mi?: {grav_results["poisson_model"]["overdispersion_detected"]}")
print(f"Önerilen Model: {grav_results["synthesis"]["primary_model_recommendation"]}")
print(f"\nBilimsel Sonuç: {grav_results["synthesis"]["scientific_conclusion"]["distance_effect"]}")
print(f"Ege Bölgesi Etkisi: {grav_results["synthesis"]["scientific_conclusion"]["egean_effect"]}")

## 7. Türkiye Coğrafi İş Birliği Haritası
Üretilen etkileşimli harita `reports/generated/turkey_collaboration_map.html` dosyası olarak kaydedilmiştir. Folium tabanlı bu haritada şehir içi koordinat çakışmaları açısal dağıtım (jitter) ile çözülmüş, düğüm boyutları derece ile ölçeklendirilmiş ve güçlü iş birlikleri ($w \ge 5$) görselleştirilmiştir.

In [ ]:
print("Harita dosya yolu: reports/generated/turkey_collaboration_map.html")
print("Harita boyutu:", os.path.getsize("../reports/generated/turkey_collaboration_map.html"), "bayt")
# Notebook içinde küçük bir iframe ile gösterme seçeneği:
HTML("<iframe src=""../reports/generated/turkey_collaboration_map.html"" width=""100%"" height=""500px""></iframe>")